### 1. Libraries & Function


In [2]:
import os
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from oemof.solph import (Bus, EnergySystem, Flow, Model, create_time_index, processing)
from oemof.solph.components import (Sink, Source, Converter, GenericStorage)
from oemof.solph import EnergySystem
from oemof.solph import views
import oemof.solph as solph


def LCOH(invest_cost, operation_cost, heat_produced, revenue=0, i=0.05, n=20):
    pvf = ((1 + i) ** n - 1) / ((1 + i) ** n * i)
    return (invest_cost + pvf * (operation_cost - revenue)) / (
        pvf * heat_produced
    )
def epc(invest_cost, i=0.05, n=20):
    af = (i * (1 + i) ** n) / ((1 + i) ** n - 1)
    return invest_cost * af

filename = r"OEMOF_files/input_data.csv"
data = pd.read_csv(filename)
solver = "gurobi"
solver_verbose = False

print(data.columns.tolist())

['thermal_demand_MW', 'electricity_price_Euro_Per_MWh', 'co2_price', 'gas_price_Euro_Per_MWh', 'biomass_Euro_Per_MWh', 'waste_Euro_Per_MWh', 'irradiance_direct_MW/m2', 'irradiance_diffuse_MW/m2']


In [3]:
from oemof.solph import processing as solph_processing

def set_result_index_safe(df_dict, k, result_index):
    """Drop-in replacement for oemof.solph.processing.set_result_index
    that does not call .label on None and shows a clearer error."""
    try:
        df_dict[k].index = result_index
    except ValueError as e:
        # Fallback: just show the raw key, no .label
        raise ValueError(f"{e}  (while setting time index for key {k})")

# overwrite the original
solph_processing.set_result_index = set_result_index_safe


### Add Losses

In [4]:
import pandas as pd

# 1. Read the CSV file
heat_losses = pd.read_csv("./Outputs/Ex1.results_edges.csv")

# 2. Inspect the columns
print("Columns in the dataset:")
print(heat_losses.columns)

# 3. Sum the losses column
total_capacity = heat_losses["capacity"].sum()
total_losses = heat_losses["losses"].sum()
losses_percentage = (total_losses / total_capacity)
efficiency = 1 - losses_percentage

print("Total capacity:", total_capacity)
print("Total losses:", total_losses)
print("Losses percentage:", losses_percentage * 100, "%")
print("Efficiency after losses:", efficiency * 100, "%")

#Add the losses to thermal demand
data["thermal_demand_MW"] = data["thermal_demand_MW"]/efficiency
data["thermal_demand_MW"].describe()


Columns in the dataset:
Index(['id', 'from_node', 'to_node', 'length', 'hp_type', 'capacity',
       'direction', 'costs', 'losses'],
      dtype='object')
Total capacity: 30710.658423
Total losses: 519.5669885615306
Losses percentage: 1.6918132506479038 %
Efficiency after losses: 98.3081867493521 %


count    8760.000000
mean        3.891914
std         1.922748
min         1.349698
25%         2.018726
50%         3.759638
75%         5.356320
max         9.422105
Name: thermal_demand_MW, dtype: float64

In [5]:

# === TIME INDEX: FIXED TO MATCH len(data) EXACTLY (NO create_time_index) === #
#datetimeindex = pd.date_range(
 #   start="2023-01-01 00:00:00",
  #  periods=len(data),   # e.g. 8760 hours
   # freq="H",
#)

# optional but highly recommended: give data the same index
#data = data.set_index(datetimeindex)

#energysystem = EnergySystem(timeindex=datetimeindex, infer_last_interval=False)

#neg_hours = data["electricity_price_Euro_Per_MWh"] < 0
#data.loc[neg_hours, "electricity_price_Euro_Per_MWh"] = 0



### 2. Energy System

In [6]:
# ------------------------------------------------------------------
# ASSUMPTIONS / INPUTS
# ------------------------------------------------------------------
# Assumes you already have a pandas DataFrame called `data`
# with columns:
#   "gas_price_Euro_Per_MWh"
#   "electricity_price_Euro_Per_MWh"
#   "biomass_Euro_Per_MWh"
#   "waste_Euro_Per_MWh"
#   "thermal_demand_MW"
#   "irradiance_direct_MW/m2"
#   "irradiance_diffuse_MW/m2"

# Global overall efficiency used for scaling capacities
#efficiency = 0.9  # <-- set this to your intended value

capacity_gas_heat    = 5 #/ efficiency   # MW
capacity_biomass_chp = 15 #/ efficiency  # MW
capacity_wte_chp     = 7 #/ efficiency   # MW
max_heat_hp          = 3                # MW
area_m2              = 20000            # m2

# ------------------------------------------------------------------
# TIME INDEX & ENERGY SYSTEM
# ------------------------------------------------------------------
datetimeindex = create_time_index(2023, number=len(data))
energysystem = EnergySystem(timeindex=datetimeindex, infer_last_interval=False)

electrical_bus = Bus(label="electrical_bus")   # Electricity bus
thermal_bus    = Bus(label="thermal_bus")      # Heat bus
gas_bus        = Bus(label="gas_bus")          # Natural gas bus
biomass_bus    = Bus(label="biomass_bus")      # Biomass bus
waste_bus      = Bus(label="waste_bus")        # Municipal Waste bus
wasteheat_bus  = Bus(label="wasteheat_bus")    # Waste Heat bus

energysystem.add(
    electrical_bus,
    thermal_bus,
    gas_bus,
    biomass_bus,
    waste_bus,
    wasteheat_bus,
)

# ------------------------------------------------------------------
# SOURCES
# ------------------------------------------------------------------
energysystem.add(
    Source(
        label="natural_gas",
        outputs={gas_bus: Flow(variable_costs=data["gas_price_Euro_Per_MWh"])},
    )
)

energysystem.add(
    Source(
        label="electricity_grid",
        outputs={electrical_bus: Flow(variable_costs=data["electricity_price_Euro_Per_MWh"])},
    )
)

energysystem.add(
    Source(
        label="biomass",
        outputs={biomass_bus: Flow(variable_costs=data["biomass_Euro_Per_MWh"])},
    )
)

energysystem.add(
    Source(
        label="waste",
        outputs={waste_bus: Flow(variable_costs=data["waste_Euro_Per_MWh"])},
    )
)

energysystem.add(
    Source(
        label="waste_heat",
        outputs={wasteheat_bus: Flow(variable_costs=0, nominal_value=0.5, max=1.0)},
    )
)

# ------------------------------------------------------------------
# LOADS
# ------------------------------------------------------------------
thermal_peak = data["thermal_demand_MW"].max()

energysystem.add(
    Sink(
        label="thermal_demand",
        inputs={
            thermal_bus: Flow(
                nominal_value=thermal_peak,
                fix=data["thermal_demand_MW"] / thermal_peak,
            )
        },
    )
)

energysystem.add(
    Sink(
        label="excess_electricity",
        inputs={
            electrical_bus: Flow(
                variable_costs=data["electricity_price_Euro_Per_MWh"] * -1
            )
        },
    )
)

# ------------------------------------------------------------------
# CONVERTERS
# 1) Natural Gas DH Only Unit
# ------------------------------------------------------------------
efficiency_gas_heat = 0.93
startup_cost_gas = 8.0   # €/start
vom_heat = 0.64          # €/MWh_heat

energysystem.add(
    solph.components.Converter(
        label="heat_gas",
        inputs={gas_bus: solph.Flow()},
        outputs={
            thermal_bus: solph.Flow(
                nominal_value=capacity_gas_heat,
                min=0.1,
                variable_costs=vom_heat,   # O&M only
                nonconvex=solph.NonConvex(
                    startup_costs=startup_cost_gas,
                    shutdown_costs=0,
                    # minimum_uptime=1,
                    # minimum_downtime=1,
                    # initial_status=0,
                ),
            )
        },
        conversion_factors={thermal_bus: efficiency_gas_heat},
    )
)

#from oemof import solph
from oemof.solph import Flow

# ------------------------------------------------------------------
#2. Biomass CHP parameters
# ------------------------------------------------------------------

# --- efficiencies from your datasheet ---
efficiency_biomass_chp_elc = 0.14        # net electric efficiency (CHP mode)
efficiency_biomass_chp_th_rel = 0.97     # "97% of remaining energy"
efficiency_biomass_chp_th = (
    1 - efficiency_biomass_chp_elc
) * efficiency_biomass_chp_th_rel        # ≈ 0.83 absolute heat efficiency

eta_el_cond_bio = 0.20   # assumed condensing electric efficiency (no heat)

# ---------- derive start-up / shutdown costs from your data ----------
eta_tot_biomass = efficiency_biomass_chp_elc + efficiency_biomass_chp_th  # ≈ 0.97
avg_biomass_price = float(data["biomass_Euro_Per_MWh"].mean())            # €/MWh_fuel

# fuel input at full CHP load (MW_fuel)
fuel_input_full = capacity_biomass_chp / eta_tot_biomass

# variable cost per hour at full load (€/h)
C_var_full = fuel_input_full * avg_biomass_price

# assume "equivalent hours" of variable cost per start
equiv_hours = 4.0
biomass_startup_cost = equiv_hours * C_var_full     # €/start
biomass_shutdown_cost = 0.2 * biomass_startup_cost  # €/shutdown

# you can use these also for a separate biomass boiler if needed
boiler_startup_cost  = biomass_startup_cost
boiler_shutdown_cost = biomass_shutdown_cost

# ---------- non-convex behaviour settings ----------
biomass_min_load     = 0.6    # ≥ 60% of fuel capacity when ON
biomass_min_uptime   = 96     # hours
biomass_min_downtime = 48     # hours

# ------------------------------------------------------------------
# Biomass CHP as ExtractionTurbineCHP
# ------------------------------------------------------------------

energysystem.add(
    solph.components.ExtractionTurbineCHP(
        label="chp_biomass",
        inputs={
            biomass_bus: Flow(
                nominal_value=capacity_biomass_chp,   # MW_fuel capacity
                min=biomass_min_load,                 # min load when ON
                nonconvex=solph.NonConvex(
                    startup_costs=biomass_startup_cost,
                    shutdown_costs=biomass_shutdown_cost,
                    minimum_uptime=biomass_min_uptime,
                    minimum_downtime=biomass_min_downtime,
                    initial_status=1,
                ),
            )
        },
        outputs={
            electrical_bus: Flow(),
            thermal_bus:    Flow(),
        },
        # CHP-mode efficiencies (back-pressure / extraction region)
        conversion_factors={
            electrical_bus: efficiency_biomass_chp_elc,
            thermal_bus:    efficiency_biomass_chp_th,
        },
        # Condensing-mode electric efficiency (no heat)
        conversion_factor_full_condensation={
            electrical_bus: eta_el_cond_bio,
        },
    )
)

from oemof import solph
from oemof.solph import Flow

# ------------------------------------------------------------------
# Waste-to-Energy (WtE) CHP parameters
# ------------------------------------------------------------------

# --- efficiencies from your datasheet ---
efficiency_wte_chp_elc = 0.22          # net electric efficiency (CHP mode)
efficiency_wte_chp_th_rel = 0.80       # "80% of remaining energy"
efficiency_wte_chp_th = (
    1 - efficiency_wte_chp_elc
) * efficiency_wte_chp_th_rel          # ≈ 0.624 absolute heat efficiency

# assumed condensing electric efficiency (no heat)
eta_el_cond_wte = 0.26                 # tune if you have better data

# ---------- derive start-up / shutdown costs from your data ----------
# total CHP efficiency
eta_tot_wte = efficiency_wte_chp_elc + efficiency_wte_chp_th

# average fuel (waste) price in €/MWh_fuel
# NOTE: make sure this column exists in your `data` DataFrame
avg_wte_price = float(data["waste_Euro_Per_MWh"].mean())

# fuel input at full CHP load (MW_fuel)
fuel_input_full_wte = capacity_wte_chp / eta_tot_wte

# variable cost per hour at full load (€/h)
C_var_full_wte = fuel_input_full_wte * avg_wte_price

# assume "equivalent hours" of variable cost per start
equiv_hours_wte = 4.0
wte_startup_cost = equiv_hours_wte * C_var_full_wte      # €/start
wte_shutdown_cost = 0.2 * wte_startup_cost               # €/shutdown

# ---------- non-convex behaviour settings ----------
wte_min_load     = 0.5    # ≥ 50% of fuel capacity when ON
wte_min_uptime   = 48     # hours
wte_min_downtime = 24     # hours

# ------------------------------------------------------------------
# Waste-to-Energy CHP as ExtractionTurbineCHP
# ------------------------------------------------------------------

energysystem.add(
    solph.components.ExtractionTurbineCHP(
        label="waste_to_energy",
        inputs={
            waste_bus: Flow(
                nominal_value=capacity_wte_chp,    # MW_fuel capacity
                min=wte_min_load,                  # minimum load when ON
                nonconvex=solph.NonConvex(
                    startup_costs=wte_startup_cost,
                    shutdown_costs=wte_shutdown_cost,
                    minimum_uptime=wte_min_uptime,
                    minimum_downtime=wte_min_downtime,
                    initial_status=1,
                ),
            )
        },
        outputs={
            electrical_bus: Flow(),
            thermal_bus:    Flow(),
        },
        # CHP-mode efficiencies (back-pressure / extraction region)
        conversion_factors={
            electrical_bus: efficiency_wte_chp_elc,
            thermal_bus:    efficiency_wte_chp_th,
        },
        # Condensing-mode electric efficiency (no heat)
        conversion_factor_full_condensation={
            electrical_bus: eta_el_cond_wte,
        },
    )
)


# ------------------------------------------------------------------
# 4) Heat Pump
# ------------------------------------------------------------------
cop_hp = 3.5

energysystem.add(
    solph.components.Converter(
        label="heat_pump",
        inputs={
            electrical_bus: Flow(),  # electricity input
            wasteheat_bus: Flow(),   # low-temperature/source heat
        },
        outputs={
            thermal_bus: Flow(nominal_value=max_heat_hp),  # max thermal capacity [MW]
        },
        conversion_factors={
            electrical_bus: 1 / cop_hp,              # electrical input
            wasteheat_bus: (cop_hp - 1) / cop_hp,    # source heat input
        },
    )
)

# ------------------------------------------------------------------
# 5) Solar Thermal Plant
# ------------------------------------------------------------------
tilt_deg = 45              # collector tilt
eta = 0.48                 # annual efficiency
annual_input = 1.05        # MWh_i per m2 per year
pump_fraction = 0.01       # 1% parasitic load

dni = data["irradiance_direct_MW/m2"].clip(lower=0)
dhi = data["irradiance_diffuse_MW/m2"].clip(lower=0)
cosZ = 0.3                 # simple average factor
ghi = dhi + dni * cosZ

tilt = np.deg2rad(tilt_deg)
Rb = 1.0                   # simple assumption (no tracking)

G_tilt = (
    dni * Rb +                                # beam on tilt
    dhi * (1 + np.cos(tilt)) / 2 +           # diffuse on tilt
    ghi * 0.2 * (1 - np.cos(tilt)) / 2       # ground reflection (albedo = 0.2)
)

G_tilt = G_tilt.clip(lower=0)
G_MWh = G_tilt / 1e6                          # W/m2 → MWh/m2 over 1h

scale = annual_input / G_MWh.sum()
G_input = G_MWh * scale                       # MWh_i/m2/h (scaled)
G_output = G_input * eta                      # MWh_th/m2/h

solar_thermal_MW = G_output * area_m2         # MWh/h ≈ MW
pump_MW = pump_fraction * solar_thermal_MW

print("Solar thermal peak (MW):", solar_thermal_MW.max())
print("Annual thermal output (MWh):", solar_thermal_MW.sum())

Q_nominal = solar_thermal_MW.max()
solar_profile_pu = solar_thermal_MW / Q_nominal

energysystem.add(
    Source(
        label="solar_thermal",
        outputs={
            thermal_bus: Flow(
                nominal_value=Q_nominal,
                max=solar_profile_pu.values,
            )
        },
    )
)

# ------------------------------------------------------------------
# HEAT STORAGE
# ------------------------------------------------------------------
storage_capacity_MWh = 80   # max stored heat [MWh]
max_charge_MW = 10          # max charging power [MW]
max_discharge_MW = 10       # max discharging power [MW]
loss_rate = 0.002           # fraction of energy lost per hour
initial_soc = 0.2           # 20% full at t=0
eff_charge = 0.98           # charging efficiency
eff_discharge = 0.98        # discharging efficiency

energysystem.add(
    GenericStorage(
        label="heat_storage",
        inputs={
            thermal_bus: Flow(
                nominal_value=max_charge_MW,
                max=1.0,   # 0..1 * nominal_value
            )
        },
        outputs={
            thermal_bus: Flow(
                nominal_value=max_discharge_MW,
                max=1.0,   # 0..1 * nominal_value
            )
        },
        nominal_storage_capacity=storage_capacity_MWh,
        loss_rate=loss_rate,             # standing losses per timestep
        initial_storage_level=initial_soc,  # relative (0..1)
        inflow_conversion_factor=eff_charge,
        outflow_conversion_factor=eff_discharge,
        min_storage_level=0.2,
        max_storage_level=0.8,
    )
)

# ------------------------------------------------------------------
# OPTIMIZE
# ------------------------------------------------------------------
model = Model(energysystem, name="District Heating Optimization Model")
logging.info("Solving the optimization problem.")

model.solve(
    solver="gurobi",
    solve_kwargs={"tee": True},
    cmdline_options={"ratioGap": "0.30"},
)

energysystem.results["main"] = processing.results(model, remove_last_time_point=True)
#energysystem.results["main"] = processing.results(model)
energysystem.results["meta"] = processing.meta_results(model)

output_file = os.path.join(os.getcwd(), "Outputs/results.xlsx")
energysystem.dump(os.getcwd(), "Outputs/results.xlsx")
logging.info("Results have been dumped.")

results = energysystem.results["main"]

# quick views
electrical_bus_view = views.node(results, "electrical_bus")
thermal_bus_view    = views.node(results, "thermal_bus")
gas_bus_view        = views.node(results, "gas_bus")
biomass_bus_view    = views.node(results, "biomass_bus")


Solar thermal peak (MW): 8.43838564934685
Annual thermal output (MWh): 10079.999999999998
(type=<class 'pyomo.core.base.expression.ScalarExpression'>) on block
NonConvexFlowBlock with a new Component (type=<class
'pyomo.core.base.expression.ScalarExpression'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
(type=<class 'pyomo.core.base.expression.ScalarExpression'>) on block
NonConvexFlowBlock with a new Component (type=<class
'pyomo.core.base.expression.ScalarExpression'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
Set parameter Username
Set parameter LicenseID to value 2729760
Academic license - for non-commercial use only - expires 2026-10-30
Traceback (most recent call last):
  File "<stdin>", line 4, in <module>
  File "c:\Users\ardia\miniconda3\envs\env_P2\lib\site-packages\pyomo\solvers\plugins\solvers\GUROBI_RUN.py", l

ERROR:pyomo.opt:Solver (gurobi) returned non-zero return code (1)
ERROR:pyomo.opt:See the solver log above for diagnostic information.


ApplicationError: Solver (gurobi) did not exit normally

In [ ]:
print(thermal_bus_view)

In [ ]:
import matplotlib.pyplot as plt
from oemof.solph import views

# --------- THERMAL BUS RESULTS ---------
thermal_bus_view = views.node(results, "thermal_bus")
seq = thermal_bus_view["sequences"].iloc[:-1]  # drop last NaN row

start = "2023-07-01"
end   = "2023-07-08"
seq = seq.loc[start:end]

# Column names from your results
col_wte       = (("waste_to_energy", "thermal_bus"), "flow")
col_biomass   = (("chp_biomass",     "thermal_bus"), "flow")
col_gas       = (("heat_gas",        "thermal_bus"), "flow")
col_hp        = (("heat_pump",       "thermal_bus"), "flow")
col_solar_th  = (("solar_thermal",   "thermal_bus"), "flow")

# Storage flows (bus ↔ storage)
col_storage_discharge = (("heat_storage", "thermal_bus"), "flow")   # discharge to bus
col_storage_charge    = (("thermal_bus", "heat_storage"), "flow")   # charge from bus

has_hp          = col_hp in seq.columns
has_solar_th    = col_solar_th in seq.columns
has_storage_dis = col_storage_discharge in seq.columns
has_storage_ch  = col_storage_charge in seq.columns

# --------- STACKED SUPPLY COMPONENTS ---------
stack_order = [
    "biomass",
    "wte",
    "hp",
    "gas",
    "storage",
    "solar_th",
]

components = {
    "biomass": {
        "series": seq[col_biomass],
        "label": "Biomass CHP",
        "color": "#8AB020",
    },
    "gas": {
        "series": seq[col_gas],
        "label": "Gas Heat Unit",
        "color": "#C87E00",
    },
    "wte": {
        "series": seq[col_wte],
        "label": "Waste-to-Energy",
        "color": "#008DDF",
    },
    "hp": {
        "series": seq[col_hp] if has_hp else None,
        "label": "Heat Pump",
        "color": "#F01000",
    },
    "solar_th": {
        "series": seq[col_solar_th] if has_solar_th else None,
        "label": "Solar Thermal",
        "color": "#C500C8",
    },
    "storage": {
        "series": seq[col_storage_discharge] if has_storage_dis else None,
        "label": "Storage discharge",
        "color": "#7F7F7F",
    },
}

supply_components = []
labels = []
colors = []

for key in stack_order:
    c = components[key]
    if c["series"] is not None:
        supply_components.append(c["series"])
        labels.append(c["label"])
        colors.append(c["color"])

# Total supply into thermal bus (sum of all stacked series)
total_supply = sum(supply_components)

# Storage charging from bus (0 if not present)
storage_charge = seq[col_storage_charge] if has_storage_ch else 0.0

# Net heat that actually serves demand (bus balance)
net_to_demand = total_supply - storage_charge

# --------- PLOTTING ---------
fig, ax1 = plt.subplots(figsize=(15, 6))

# STACKED SUPPLY (left axis)
ax1.stackplot(
    seq.index,
    *supply_components,
    labels=labels,
    colors=colors,
    alpha=0.7,
)

# Thermal demand
demand_series = seq[(("thermal_bus", "thermal_demand"), "flow")]
ax1.plot(
    seq.index,
    demand_series,
    label="Thermal Demand",
    color="#262630",
    linestyle="--",
    linewidth=0.9,
)

# Net heat to demand (should match demand if only demand + storage charge on bus)
ax1.plot(
    seq.index,
    net_to_demand,
    label="Net heat to demand",
    color="black",
    linewidth=1.2,
)

ax1.set_ylabel("Heat Power [MW]")
ax1.set_title("Thermal Bus: Heat Supply + Demand + Storage")
ax1.grid(True, alpha=0.3)

# --------- STORAGE SOC (right axis) ---------
storage_view = views.node(results, "heat_storage")
storage_seq = storage_view["sequences"].iloc[:-1].loc[start:end]

# Auto-detect SOC column
soc_cols = [c for c in storage_seq.columns if c[1] == "storage_content"]
soc = storage_seq[soc_cols[0]].clip(lower=0)

# Normalize by physical capacity (MWh)
soc_pu = soc / storage_capacity_MWh

ax2 = ax1.twinx()
ax2.plot(
    soc.index,
    soc_pu,
    color="black",
    linewidth=1.2,
    linestyle=":",
    label="Storage SoC",
)
ax2.set_ylabel("State of Charge [0–1]")
ax2.set_ylim(0, 1)

# --------- MERGED LEGEND ---------
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from oemof.solph import views

# --------- THERMAL BUS RESULTS ---------
thermal_bus_view = views.node(results, "thermal_bus")
seq = thermal_bus_view["sequences"].iloc[:-1]  # drop last NaN row

start = "2023-08-01"
end   = "2023-08-07"
seq = seq.loc[start:end]

# Column names from your results
col_wte       = (("waste_to_energy", "thermal_bus"), "flow")
col_biomass   = (("chp_biomass",     "thermal_bus"), "flow")
col_gas       = (("heat_gas",        "thermal_bus"), "flow")
col_hp        = (("heat_pump",       "thermal_bus"), "flow")
col_solar_th  = (("solar_thermal",   "thermal_bus"), "flow")

# Storage flows (bus ↔ storage)
col_storage_discharge = (("heat_storage", "thermal_bus"), "flow")   # discharge to bus
col_storage_charge    = (("thermal_bus", "heat_storage"), "flow")   # charge from bus

has_hp          = col_hp in seq.columns
has_solar_th    = col_solar_th in seq.columns
has_storage_dis = col_storage_discharge in seq.columns
has_storage_ch  = col_storage_charge in seq.columns

# --------- GROSS SUPPLY COMPONENTS (to bus) ---------
components = {
    "Biomass CHP":       seq[col_biomass],
    "Waste-to-Energy":   seq[col_wte],
    "Heat Pump":         seq[col_hp] if has_hp else 0.0,
    "Gas Heat Unit":     seq[col_gas],
    "Storage discharge": seq[col_storage_discharge] if has_storage_dis else 0.0,
    "Solar Thermal":     seq[col_solar_th] if has_solar_th else 0.0,
}

colors = {
    "Biomass CHP":       "#8AB020",
    "Waste-to-Energy":   "#008DDF",
    "Heat Pump":         "#F01000",
    "Gas Heat Unit":     "#C87E00",
    "Storage discharge": "#7F7F7F",
    "Solar Thermal":     "#C500C8",
}

# Total gross thermal supply into bus
total_supply = sum(components.values())

# Storage charging from bus (0 if not present)
storage_charge = seq[col_storage_charge] if has_storage_ch else 0.0

# Net heat that actually serves demand
net_to_demand = (total_supply - storage_charge).clip(lower=0)

# --------- REDISTRIBUTE NET HEAT TO TECHNOLOGIES ---------
# For each timestep, allocate net_to_demand according to each tech's share of total_supply
stack_series = []
labels = []
stack_colors = []

# avoid division by zero
denom = total_supply.replace(0, np.nan)

for name, series in components.items():
    # series may be 0.0 (scalar) if tech not present
    if isinstance(series, (int, float)):
        tech_net = total_supply * 0  # all zeros
    else:
        share = (series / denom).fillna(0)
        tech_net = share * net_to_demand
    stack_series.append(tech_net)
    labels.append(name)
    stack_colors.append(colors[name])

# Demand series
demand_series = seq[(("thermal_bus", "thermal_demand"), "flow")]

# --------- PLOTTING ---------
fig, ax1 = plt.subplots(figsize=(15, 6))

# STACKED "HEAT TO DEMAND" (left axis)
ax1.stackplot(
    seq.index,
    *stack_series,
    labels=labels,
    colors=stack_colors,
    alpha=0.7,
)

# Thermal demand – should now match the top of the stack
ax1.plot(
    seq.index,
    demand_series,
    label="Thermal Demand",
    color="#262630",
    linestyle="--",
    linewidth=0.9,
)

ax1.set_ylabel("Heat Power [MW]")
ax1.set_title("How Heat Supply Meets Demand (incl. Storage)")
ax1.grid(True, alpha=0.3)

# --------- STORAGE SOC (right axis) ---------
storage_view = views.node(results, "heat_storage")
storage_seq = storage_view["sequences"].iloc[:-1].loc[start:end]

# Auto-detect SOC column
soc_cols = [c for c in storage_seq.columns if c[1] == "storage_content"]
soc = storage_seq[soc_cols[0]].clip(lower=0)

soc_pu = soc / storage_capacity_MWh  # 0–1

ax2 = ax1.twinx()
ax2.plot(
    soc.index,
    soc_pu,
    color="black",
    linewidth=1.2,
    linestyle=":",
    label="Storage SoC",
)
ax2.set_ylabel("State of Charge [0–1]")
ax2.set_ylim(0, 1)

# --------- MERGED LEGEND ---------
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from oemof.solph import views

# Extract sequences
thermal_bus_view = views.node(results, "thermal_bus")
seq = thermal_bus_view["sequences"].iloc[:-1]  # drop last NaN row

# Ensure datetime index
if not isinstance(seq.index, pd.DatetimeIndex):
    seq.index = pd.to_datetime(seq.index)

# Define the technology columns (all thermal outputs to thermal_bus)
tech_cols = {
    "Biomass CHP":       (("chp_biomass",     "thermal_bus"), "flow"),
    "Gas Heat Unit":     (("heat_gas",        "thermal_bus"), "flow"),
    "Waste-to-Energy":   (("waste_to_energy", "thermal_bus"), "flow"),
    "Heat Pump":         (("heat_pump",       "thermal_bus"), "flow"),
    "Solar Thermal":     (("solar_thermal",   "thermal_bus"), "flow"),
    # NEW: storage discharge to thermal bus
    "Storage discharge": (("heat_storage",    "thermal_bus"), "flow"),
}

# Aggregate annual production by technology
annual_yields = {}
for tech, col in tech_cols.items():
    if col in seq.columns:
        annual_sum = seq[col].resample("Y").sum().iloc[0]  # first year only
        annual_yields[tech] = annual_sum
    else:
        annual_yields[tech] = 0.0  # tech not used or not in results

# Prepare for bar chart
technologies = list(annual_yields.keys())
values = [annual_yields[tech] for tech in technologies]

plt.figure(figsize=(10, 6))
bars = plt.bar(
    technologies,
    values,
    color=["forestgreen", "firebrick", "slateblue", "darkorange", "goldenrod", "gray"],
)
plt.xlabel("Technology", fontsize=12)
plt.ylabel("Annual Production [MWh/year]", fontsize=12)
plt.title("Annual Thermal Supply to Network by Technology", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from pyomo.opt import TerminationCondition, SolverStatus

meta = energysystem.results["meta"]

print("\n==================== META RESULTS ====================\n")

# Objective
print(f"Objective value             : {float(meta['objective']):,.3f}")

# Problem information
problem = meta.get("problem", {})
print("\n--- Problem Structure ---")
print(f"Name                        : {problem.get('Name')}")
print(f"Constraints                 : {problem.get('Number of constraints'):,}")
print(f"Variables                   : {problem.get('Number of variables'):,}")
print(f"Nonzeros                    : {problem.get('Number of nonzeros'):,}")
print(f"Sense                       : {problem.get('Sense').name}")

# Solver information
solver = meta.get("solver", {})
print("\n--- Solver Information ---")
print(f"Status                      : {solver.get('Status').name}")
print(f"Termination condition       : {solver.get('Termination condition').name}")
print(f"Termination message         : {solver.get('Termination message')}")
print(f"User time (s)               : {solver.get('User time')}")
print(f"System time (s)             : {solver.get('System time')}")
print(f"Wallclock time (s)          : {solver.get('Wallclock time')}")
print(f"Reported Time (Pyomo) (s)   : {solver.get('Time')}")

print("\n======================================================\n")
